#1. Parameters

In [0]:
dbutils.widgets.dropdown(
    "taxi_type",
    "yellow",
    ["yellow", "green"],
    "Taxi type"
)

dbutils.widgets.text("year", "2025", "Year")
dbutils.widgets.text("month", "1", "Month")

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

taxi_type = dbutils.widgets.get("taxi_type").strip().lower()
year = int(dbutils.widgets.get("year"))
month = int(dbutils.widgets.get("month"))

if taxi_type not in {"yellow", "green"}:
    raise ValueError("taxi_type must be yellow or green.")

if year < 2009:
    raise ValueError("year cannot be earlier than 2009.")

if not 1 <= month <= 12:
    raise ValueError("month must be between 1 and 12.")

now_utc = datetime.now(timezone.utc)

if (year, month) > (now_utc.year, now_utc.month):
    raise ValueError("The requested period is in the future.")

gold_run_id = uuid4().hex
gold_started_at = datetime.now(timezone.utc)

print(f"Gold run ID: {gold_run_id}")
print(f"Processing: {taxi_type}/{year}/{month:02d}")

In [0]:
from pyspark.sql import functions as F

silver_table = "workspace.urban_mobility_silver.taxi_trips"

silver_partition_df = (
    spark.table(silver_table).filter(F.col("taxi_type") == taxi_type)
    .filter(F.col("year") == year)
    .filter(F.col("month") == month)
    .select("pickup_datetime",
            "dropoff_datetime",
            "passenger_count",
            "trip_distance",
            "pickup_location_id",
            "fare_amount",
            "tip_amount",
            "total_amount",
            "taxi_type",
            "year",
            "month")
    )



input_rows = silver_partition_df.count()

if input_rows == 0:
    raise ValueError(
        f"No Silver records found for {taxi_type}/{year}/{month:02d}."
    )

print(f"Silver input rows: {input_rows:,}")



In [0]:
enriched_df = (
    silver_partition_df
    .withColumn(
        "pickup_date",
        F.to_date("pickup_datetime")
    )
    .withColumn(
        "trip_duration_minutes",
        (
            F.unix_timestamp("dropoff_datetime")
            - F.unix_timestamp("pickup_datetime")
        ) / 60.0
    )
    .withColumn(
        "tip_percentage",                     
        F.when(F.col("fare_amount") > 0,
               F.col("tip_amount")/F.col("fare_amount")*100 ).otherwise(F.lit("0.00"))
    )
    .withColumn(
        "average_speed_mph",
        F.when(F.col("trip_duration_minutes") > 0, F.col("trip_distance")/(F.col("trip_duration_minutes")/60.0 )).otherwise(F.lit(None).cast("double"))
        )
    )
enriched_df.select(
    "pickup_date",
    "trip_duration_minutes",
    "tip_percentage",
    "average_speed_mph",
).show(10, truncate=False)

#2. Daily Zone Metric

In [0]:
daily_zone_metrics_df = (
    enriched_df
    .groupBy(
        "taxi_type",
        "year",
        "month",
        "pickup_date",
        "pickup_location_id",
    )
    .agg(
        F.count("*").alias("trip_count"),

        F.sum(
            F.coalesce("passenger_count", F.lit(0))
        ).alias("total_passengers"),

        F.round(
            F.sum("trip_distance"), 2
        ).alias("total_distance_miles"),

        F.round(
            F.avg("trip_distance"), 2
        ).alias("average_trip_distance_miles"),

        F.round(
            F.sum("fare_amount"), 2
        ).alias("total_fare_amount"),

        F.round(
            F.sum("tip_amount"), 2
        ).alias("total_tip_amount"),

        F.round(
            F.sum("total_amount"), 2
        ).alias("total_revenue"),

        F.round(
            F.avg("total_amount"), 2
        ).alias("average_trip_value"),

        F.round(
            F.avg("trip_duration_minutes"), 2
        ).alias("average_trip_duration_minutes"),

        F.round(
            F.sum("trip_duration_minutes"), 2
        ).alias("total_duration_minutes"),
    )
)

In [0]:
daily_zone_metrics_df = (
    daily_zone_metrics_df
    .withColumn(
        "tip_percentage",
        F.when(
            F.col("total_fare_amount") > 0,
            F.round(
                F.col("total_tip_amount")
                / F.col("total_fare_amount")
                * 100,
                2,
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "average_speed_mph",
        F.when(
            F.col("total_duration_minutes") > 0,
            F.round(
                F.col("total_distance_miles")
                / (F.col("total_duration_minutes") / 60),
                2,
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

In [0]:
gold_rows = daily_zone_metrics_df.count()

print(f"Gold rows: {gold_rows:,}")

display(
    daily_zone_metrics_df
    .orderBy(
        F.desc("trip_count")
    )
    .limit(20)
)

In [0]:
invalid_gold_rows = (
    daily_zone_metrics_df
    .filter(
        F.col("pickup_date").isNull()
        | F.col("pickup_location_id").isNull()
        | (F.col("trip_count") <= 0)
        | (F.col("total_revenue") < 0)
    )
    .count()
)

if invalid_gold_rows > 0:
    raise ValueError(
        f"Gold validation failed: {invalid_gold_rows} invalid rows."
    )

aggregated_trip_count = (
    daily_zone_metrics_df
    .agg(F.sum("trip_count").alias("trip_count"))
    .first()["trip_count"]
)

if aggregated_trip_count != input_rows:
    raise ValueError(
        "Gold reconciliation failed: "
        f"input_rows={input_rows}, "
        f"aggregated_trip_count={aggregated_trip_count}"
    )

print("Gold validation passed.")
print(f"Input trips reconciled: {aggregated_trip_count:,}")

In [0]:
gold_table = (
    "workspace.urban_mobility_gold.daily_zone_metrics"
)

partition_columns = [
    "taxi_type",
    "year",
    "month",
]

replace_predicate = (
    f"taxi_type = '{taxi_type}' "
    f"AND year = {year} "
    f"AND month = {month}"
)

print(f"Gold table: {gold_table}")
print(f"Replace predicate: {replace_predicate}")

In [0]:
gold_writer = (
    daily_zone_metrics_df
    .write
    .format("delta")
)

if spark.catalog.tableExists(gold_table):
    (
        gold_writer
        .mode("overwrite")
        .option("replaceWhere", replace_predicate)
        .saveAsTable(gold_table)
    )

    print(
        f"Replaced Gold partition: "
        f"{taxi_type}/{year}/{month:02d}"
    )

else:
    (
        gold_writer
        .mode("overwrite")
        .partitionBy(*partition_columns)
        .saveAsTable(gold_table)
    )

    print(f"Created Gold table: {gold_table}")

In [0]:
written_partition_df = (
    spark.table(gold_table)
    .filter(
        (F.col("taxi_type") == taxi_type)
        & (F.col("year") == year)
        & (F.col("month") == month)
    )
)

written_gold_rows = written_partition_df.count()

written_trip_count = (
    written_partition_df
    .agg(
        F.sum("trip_count").alias("trip_count")
    )
    .first()["trip_count"]
)

if written_gold_rows != gold_rows:
    raise ValueError(
        "Gold row-count validation failed: "
        f"expected={gold_rows}, "
        f"written={written_gold_rows}"
    )

if written_trip_count != input_rows:
    raise ValueError(
        "Written Gold reconciliation failed: "
        f"input={input_rows}, "
        f"written_trip_count={written_trip_count}"
    )

print(f"Written Gold rows: {written_gold_rows:,}")
print(f"Written trips reconciled: {written_trip_count:,}")
print("Gold write validation passed.")

In [0]:
%sql
DESCRIBE HISTORY workspace.urban_mobility_gold.daily_zone_metrics;

#3. Hourly Demand Metric

In [0]:
hourly_source_df = (
    enriched_df
    .withColumn(
        "pickup_hour",
        F.hour("pickup_datetime")
    )
    .withColumn(
        "day_of_week",
        F.date_format("pickup_datetime", "EEEE")
    )
    .withColumn(
        "day_of_week_number",
        F.dayofweek("pickup_datetime")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("pickup_datetime").isin(1, 7)
    )
)

display(hourly_source_df)

In [0]:
hourly_demand_metrics_df = (
    hourly_source_df
    .groupBy(
        "taxi_type",
        "year",
        "month",
        "pickup_date",
        "pickup_hour",
        "day_of_week",
        "day_of_week_number",
        "is_weekend",
    )
    .agg(
        F.count("*").alias("trip_count"),

        F.sum(
            F.coalesce("passenger_count", F.lit(0))
        ).alias("total_passengers"),

        F.round(
            F.sum("trip_distance"), 2
        ).alias("total_distance_miles"),

        F.round(
            F.avg("trip_distance"), 2
        ).alias("average_trip_distance_miles"),

        F.round(
            F.sum("total_amount"), 2
        ).alias("total_revenue"),

        F.round(
            F.avg("total_amount"), 2
        ).alias("average_trip_value"),

        F.round(
            F.avg("trip_duration_minutes"), 2
        ).alias("average_trip_duration_minutes"),
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

In [0]:
hourly_gold_rows = hourly_demand_metrics_df.count()

hourly_trip_count = (
    hourly_demand_metrics_df
    .agg(
        F.sum("trip_count").alias("trip_count")
    )
    .first()["trip_count"]
)

invalid_hourly_rows = (
    hourly_demand_metrics_df
    .filter(
        F.col("pickup_date").isNull()
        | ~F.col("pickup_hour").between(0, 23)
        | (F.col("trip_count") <= 0)
        | (F.col("total_revenue") < 0)
    )
    .count()
)

if invalid_hourly_rows > 0:
    raise ValueError(
        f"Hourly Gold contains {invalid_hourly_rows} invalid rows."
    )

if hourly_trip_count != input_rows:
    raise ValueError(
        "Hourly Gold reconciliation failed: "
        f"input={input_rows}, aggregated={hourly_trip_count}"
    )

print(f"Hourly Gold rows: {hourly_gold_rows:,}")
print(f"Hourly trips reconciled: {hourly_trip_count:,}")
print("Hourly Gold validation passed.")

In [0]:
hourly_gold_table = (
    "workspace.urban_mobility_gold.hourly_demand_metrics"
)

hourly_writer = (
    hourly_demand_metrics_df
    .write
    .format("delta")
)

if spark.catalog.tableExists(hourly_gold_table):
    (
        hourly_writer
        .mode("overwrite")
        .option("replaceWhere", replace_predicate)
        .saveAsTable(hourly_gold_table)
    )

    print(
        f"Replaced hourly Gold partition: "
        f"{taxi_type}/{year}/{month:02d}"
    )

else:
    (
        hourly_writer
        .mode("overwrite")
        .partitionBy(*partition_columns)
        .saveAsTable(hourly_gold_table)
    )

    print(f"Created hourly Gold table: {hourly_gold_table}")

In [0]:
written_hourly_df = (
    spark.table(hourly_gold_table)
    .filter(
        (F.col("taxi_type") == taxi_type)
        & (F.col("year") == year)
        & (F.col("month") == month)
    )
)

written_hourly_rows = written_hourly_df.count()

written_hourly_trips = (
    written_hourly_df
    .agg(F.sum("trip_count").alias("trip_count"))
    .first()["trip_count"]
)

if written_hourly_rows != hourly_gold_rows:
    raise ValueError(
        "Hourly Gold row-count validation failed."
    )

if written_hourly_trips != input_rows:
    raise ValueError(
        "Written hourly Gold reconciliation failed."
    )

print(f"Written hourly rows: {written_hourly_rows:,}")
print(f"Written hourly trips: {written_hourly_trips:,}")
print("Hourly Gold write validation passed.")

In [0]:
%sql
SELECT           
    pickup_hour,
    SUM(trip_count) AS trips,
    ROUND(SUM(total_revenue), 2) AS revenue
FROM workspace.urban_mobility_gold.hourly_demand_metrics
WHERE taxi_type = 'yellow'
  AND year = 2025
  AND month = 1
GROUP BY pickup_hour
ORDER BY trips DESC;

#Monthly Kpis

In [0]:
monthly_kpis_df = (
    enriched_df
    .groupBy(
        "taxi_type",
        "year",
        "month",
    )
    .agg(
        F.count("*").alias("total_trips"),

        F.sum(
            F.coalesce("passenger_count", F.lit(0))
        ).alias("total_passengers"),

        F.countDistinct(
            "pickup_location_id"
        ).alias("active_pickup_zones"),

        F.round(
            F.sum("trip_distance"), 2
        ).alias("total_distance_miles"),

        F.round(
            F.avg("trip_distance"), 2
        ).alias("average_trip_distance_miles"),

        F.round(
            F.sum("fare_amount"), 2
        ).alias("total_fare_amount"),

        F.round(
            F.sum("tip_amount"), 2
        ).alias("total_tip_amount"),

        F.round(
            F.sum("total_amount"), 2
        ).alias("total_revenue"),

        F.round(
            F.avg("total_amount"), 2
        ).alias("average_trip_value"),

        F.round(
            F.avg("trip_duration_minutes"), 2
        ).alias("average_trip_duration_minutes"),

        F.round(
            F.sum("trip_duration_minutes"), 2
        ).alias("total_duration_minutes"),
    )
    .withColumn(
        "tip_percentage",
        F.when(
            F.col("total_fare_amount") > 0,
            F.round(
                F.col("total_tip_amount")
                / F.col("total_fare_amount")
                * 100,
                2,
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "average_speed_mph",
        F.when(
            F.col("total_duration_minutes") > 0,
            F.round(
                F.col("total_distance_miles")
                / (F.col("total_duration_minutes") / 60),
                2,
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

In [0]:
monthly_rows = monthly_kpis_df.count()

if monthly_rows != 1:
    raise ValueError(
        f"Expected exactly one monthly KPI row, found {monthly_rows}."
    )

monthly_result = monthly_kpis_df.first()

if monthly_result["total_trips"] != input_rows:
    raise ValueError(
        "Monthly KPI reconciliation failed: "
        f"input={input_rows}, "
        f"monthly={monthly_result['total_trips']}"
    )

if monthly_result["total_revenue"] < 0:
    raise ValueError("Monthly total revenue cannot be negative.")

if monthly_result["active_pickup_zones"] <= 0:
    raise ValueError("Monthly active zones must be greater than zero.")

print("Monthly KPI validation passed.")
display(monthly_kpis_df)

In [0]:
monthly_gold_table = (
    "workspace.urban_mobility_gold.monthly_kpis"
)

monthly_writer = (
    monthly_kpis_df
    .write
    .format("delta")
)

if spark.catalog.tableExists(monthly_gold_table):
    (
        monthly_writer
        .mode("overwrite")
        .option("replaceWhere", replace_predicate)
        .saveAsTable(monthly_gold_table)
    )

    print(
        f"Replaced monthly KPI partition: "
        f"{taxi_type}/{year}/{month:02d}"
    )

else:
    (
        monthly_writer
        .mode("overwrite")
        .partitionBy(*partition_columns)
        .saveAsTable(monthly_gold_table)
    )

    print(f"Created monthly KPI table: {monthly_gold_table}")

In [0]:
written_monthly_df = (
    spark.table(monthly_gold_table)
    .filter(
        (F.col("taxi_type") == taxi_type)
        & (F.col("year") == year)
        & (F.col("month") == month)
    )
)

written_monthly_rows = written_monthly_df.count()
written_monthly_result = written_monthly_df.first()

if written_monthly_rows != 1:
    raise ValueError(
        f"Expected one written monthly row, found {written_monthly_rows}."
    )

if written_monthly_result["total_trips"] != input_rows:
    raise ValueError(
        "Written monthly KPI reconciliation failed."
    )

print("Monthly KPI write validation passed.")
display(written_monthly_df)

#Audit


In [0]:
%sql
CREATE TABLE IF NOT EXISTS
workspace.urban_mobility_ops.gold_aggregation_runs (
    run_id STRING NOT NULL,
    taxi_type STRING NOT NULL,
    data_year INT NOT NULL,
    data_month INT NOT NULL,
    started_at TIMESTAMP NOT NULL,
    completed_at TIMESTAMP NOT NULL,
    status STRING NOT NULL,
    input_rows BIGINT NOT NULL,
    daily_zone_rows BIGINT NOT NULL,
    hourly_demand_rows BIGINT NOT NULL,
    monthly_kpi_rows BIGINT NOT NULL,
    reconciled_trip_count BIGINT NOT NULL,
    error_message STRING
)
USING DELTA
COMMENT 'Audit history for Gold taxi aggregation runs';

In [0]:
from datetime import datetime, timezone

gold_completed_at = datetime.now(timezone.utc)
gold_status = "SUCCESS"

gold_audit_df = (
    spark.range(1)
    .select(
        F.lit(gold_run_id)
        .alias("run_id"),

        F.lit(taxi_type)
        .alias("taxi_type"),

        F.lit(year)
        .cast("int")
        .alias("data_year"),

        F.lit(month)
        .cast("int")
        .alias("data_month"),

        F.lit(gold_started_at)
        .cast("timestamp")
        .alias("started_at"),

        F.lit(gold_completed_at)
        .cast("timestamp")
        .alias("completed_at"),

        F.lit(gold_status)
        .alias("status"),

        F.lit(input_rows)
        .cast("long")
        .alias("input_rows"),

        F.lit(gold_rows)
        .cast("long")
        .alias("daily_zone_rows"),

        F.lit(hourly_gold_rows)
        .cast("long")
        .alias("hourly_demand_rows"),

        F.lit(monthly_rows)
        .cast("long")
        .alias("monthly_kpi_rows"),

        F.lit(monthly_result["total_trips"])
        .cast("long")
        .alias("reconciled_trip_count"),

        F.lit(None)
        .cast("string")
        .alias("error_message"),
    )
)

In [0]:
gold_audit_df.createOrReplaceTempView(
    "current_gold_aggregation_run"
)

spark.sql("""
MERGE INTO
    workspace.urban_mobility_ops.gold_aggregation_runs AS target
USING
    current_gold_aggregation_run AS source
ON target.run_id = source.run_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *
""")

print(f"Gold audit saved: {gold_run_id}")

In [0]:
gold_audit_result = spark.sql(f"""
SELECT
    *,
    input_rows = reconciled_trip_count
        AS counts_reconciled,
    completed_at >= started_at
        AS timestamps_valid
FROM workspace.urban_mobility_ops.gold_aggregation_runs
WHERE run_id = '{gold_run_id}'
""")

display(gold_audit_result)